# Precedence check

Section 5.

In [ ]:
import sys
sys.path.insert(0, "..")

from embodiment import config as cfg

import json
import re

import pandas as pd

VERBS = (r"saw|see|sees|seen|seeing|look(?:s|ed|ing)?|watch(?:es|ed|ing)?"
         r"|gaz(?:e|ed|es|ing)|stare[sd]?|staring|glanc(?:e|ed|es|ing)"
         r"|glimpse[sd]?|peer(?:s|ed|ing)?|beheld|behold(?:ing)?"
         r"|notice[sd]?|noticing|observ(?:e|es|ed|ing)"
         r"|glar(?:e|ed|es|ing)|eyeing|spotted|view(?:s|ed|ing)?"
         r"|scan(?:s|ned|ning)?|spy|spies|spied|spying"
         r"|witness(?:es|ed|ing)?")
NARROW = re.compile(rf"\b(?:{VERBS})\b", re.I)
BROAD = re.compile(rf"\b(?:{VERBS}|eyes?|eyed)\b", re.I)

HAPTIC = {"haptic_human_animal", "haptic_material"}
SIGHT = {"sight_human", "sight_material"}

## Per-book counts

In [ ]:
books = pd.read_csv(cfg.BOOK_COUNTS)
allow = set(cfg.ENGLISH_ORIGINALS.read_text().split()) - {""}
books = books[books["filename"].isin(allow)].drop_duplicates(subset="filename")

rows, mismatched = [], []
for fn, decade in zip(books["filename"], books["decade"]):
    stem = fn[:-4] if fn.endswith(".txt") else fn
    preds = json.loads((cfg.PREDICTIONS_DIR / f"{stem}-result.json").read_text())
    lines = (cfg.CORPUS_DIR / f"{stem}.txt").read_text(
        encoding="utf8", errors="replace").splitlines(keepends=True)
    if len(preds) != len(lines):          # predictions must align line for line
        mismatched.append(fn)
        continue
    n = sight = haptic = h_narrow = h_broad = 0
    for p, line in zip(preds, lines):
        n += 1
        label = p["label"]
        if label in SIGHT:
            sight += 1
        elif label in HAPTIC:
            haptic += 1
            h_narrow += bool(NARROW.search(line))
            h_broad += bool(BROAD.search(line))
    rows.append(dict(filename=fn, decade=decade, n=n, sight=sight,
                     haptic=haptic, h_narrow=h_narrow, h_broad=h_broad))

per_book = pd.DataFrame(rows)
print(f"{len(per_book):,} books, {len(mismatched)} skipped for length mismatch")

## Whole corpus

In [ ]:
t = per_book[["n", "sight", "haptic", "h_narrow", "h_broad"]].sum()
pd.DataFrame({
    "touch with looking word": [t.h_narrow, t.h_broad],
    "share of touch": [t.h_narrow / t.haptic, t.h_broad / t.haptic],
    "max gain in sight": [t.h_narrow / t.sight, t.h_broad / t.sight],
}, index=["narrow", "broad"]).round(3)

## By decade

In [ ]:
pb = per_book[per_book["decade"] >= 1810].copy()
for col in ("sight", "h_narrow", "h_broad"):
    pb[f"{col}_rate"] = pb[col] / pb["n"] * 1000

by_decade = (pb.groupby("decade")
             .agg(n_books=("filename", "count"),
                  sight_rate=("sight_rate", "mean"),
                  narrow_rate=("h_narrow_rate", "mean"),
                  broad_rate=("h_broad_rate", "mean")))
by_decade["max_gain_narrow"] = by_decade["narrow_rate"] / by_decade["sight_rate"]
by_decade["max_gain_broad"] = by_decade["broad_rate"] / by_decade["sight_rate"]
by_decade.round(3)